In [2]:
from dotenv import load_dotenv
from openai import OpenAI
from rag_helper import RAGBase
from ingest import load_data, build_index

In [3]:
load_dotenv()
openai_client = OpenAI()

In [4]:
data = load_data()
index = build_index(data)

In [5]:
instructions = """
Your task is to answer questions based on the provided context.
Use the context to find relevant information and provide accurate answers.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [6]:
assistant.rag('What is regularization')

'Regularization is a technique used to avoid overfitting by trying to make the model more simple.'

In [7]:
assistant.rag('What is reularization')

'In the provided context, **regularization** is **not defined**.\n\nIf you meant **transfer learning**, **transformer architecture**, or another term from the context, I can explain that instead.'

In [8]:
messages = [
    {"role": "user", "content": "What is regularization?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

print(response.output_text)

**Regularization** is a technique used in machine learning and statistics to **reduce overfitting** by discouraging a model from becoming too complex.

### Intuition
A very flexible model can fit the training data extremely well, including noise. Regularization adds a **penalty** or **constraint** that makes overly complicated models less attractive, helping the model generalize better to new data.

### Common forms
- **L1 regularization (Lasso):** adds a penalty proportional to the absolute value of the weights  
  \[
  \text{loss} + \lambda \sum |w_i|
  \]
  Often makes some weights exactly zero, which can lead to feature selection.

- **L2 regularization (Ridge):** adds a penalty proportional to the squared weights  
  \[
  \text{loss} + \lambda \sum w_i^2
  \]
  Tends to shrink weights smoothly toward zero.

- **Dropout:** randomly turns off some neurons during training in neural networks to prevent reliance on specific activations.

- **Early stopping:** stops training before the 

In [9]:
def search(query):
    boost_dict = {"question": 3.0}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [10]:
search('What is regulaization?')

[{'id': '22',
  'category': 'dl',
  'question': 'What is a depthwise Separable layer and what are its advantages?',
  'answer': 'Standard neural network Convolution layers involve a lot of multiplications that make them unsuitable for deployment. \n\nIn this above scenario, we have an input image of 12x12x3 pixels and we apply a 5x5 convolution(no padding, stride = 1). We stack 256 such kernels\nso that we get an output of dimensions 8x8x256.\n\nHere, there are 256 5x5x3 kernels that move 8x8 times which leads to 256x3x5x5x8x8 = 1,28,800 multiplications.\n\nDepthwise separable convolution separates this process into two parts: a depthwise convolution and a pointwise convolution.\n\nIn depthwise convolution, we apply a kernel parallelly to each channel of the image.\n\nWe end up getting 3 different outputs (representing 3 channels of the image) to get an 8x8x1 image. These are stacked together to form a 8x8x3 image.\n\nPointwise Convolution now converts this 8x8x3 image input from the d

In [11]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the ds_qa database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the ds_qa database."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [12]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
    tool_choice="required"
)

print(response.output)

[ResponseFunctionToolCall(arguments='{"query":"regularization definition machine learning overfitting penalties L1 L2"}', call_id='call_qUkWyrt1gxh8egpHL4hGK8BK', name='search', type='function_call', id='fc_0baa2835731fa4c6006a397a860108819f96f3aa8cea617fef', namespace=None, status='completed')]


In [13]:
import json


call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [15]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [21]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

print(response.output_text)

Regularization is a technique used to reduce overfitting by making a model simpler or discouraging it from learning overly large/complex parameters.

In practice, it usually adds a penalty to the loss function for large weights.

Common types:
- **L1 regularization (Lasso):** pushes some weights to exactly zero, creating sparse models.
- **L2 regularization (Ridge / weight decay):** pushes weights toward zero, but usually not exactly zero.

Why it helps:
- Reduces model complexity
- Improves generalization on unseen data
- Lowers the chance of fitting noise in the training set

If you want, I can also explain **L1 vs L2 with a simple example**.


In [25]:
usage = response.usage
input_tokens, output_tokens = usage.input_tokens, usage.output_tokens

In [26]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(input_tokens, output_tokens)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0002613


The logic behind:
1. Make a call to the LLM
2. LLM decides to call the search tool ('params')
3. We invoke the search, we get the results
4. Send the request back to the LLM (another call)
5. LLM processes the results
6. LLM gives the answer